# P2 · 25% balanced-event mixture
Screen a 25% quality/song-balanced + 75% natural mixture against the existing 0% and 50% endpoints. Enable Internet and GPU.

In [ ]:
from pathlib import Path
import subprocess, sys, os, json
project = Path("/kaggle/working/chord-practice-system")
if not project.exists():
    subprocess.run(["git", "clone", "https://github.com/PhatLavar/chord-practice-system.git", str(project)], check=True)
os.chdir(project)
subprocess.run(["git", "pull", "--ff-only", "origin", "main"], check=True)
subprocess.run([sys.executable,"-m","pip","install","--no-deps","-e","."],check=True)
print(subprocess.check_output(["git", "rev-parse", "HEAD"], text=True))
import torch, numpy
print("torch", torch.__version__, "numpy", numpy.__version__, "CUDA", torch.cuda.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
# Official inputs. Downloaded archives are read without extraction.
from urllib.request import urlretrieve
raw = project / "data/raw"
raw.mkdir(parents=True, exist_ok=True)
sources = {
    "index.csv": "https://www.dropbox.com/s/o0olz0uwl9z9stb/billboard-2.0-index.csv?dl=1",
    "full.tar.xz": "https://www.dropbox.com/s/t390alzrkx0c9yt/billboard-2.0.1-lab.tar.xz?dl=1",
    "billboard-2.0-chordino.tar.xz": "https://www.dropbox.com/s/e9dm23vbawg9dsw/billboard-2.0-chordino.tar.xz?dl=1",
}
for name, url in sources.items():
    if not (raw / name).exists():
        temporary = raw / (name + ".part")
        urlretrieve(url, temporary)
        temporary.replace(raw / name)
cache = project / "data/processed/p1"
if not (cache / "metadata.json").exists():
    subprocess.run([sys.executable, '-m', 'acr.data.billboard.prepare'], check=True)
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests"], check=True)


In [ ]:
assert torch.cuda.is_available(), "Enable a Kaggle GPU."
mixture_run = project / "runs/p2-mixture25-kaggle"
subprocess.run([sys.executable, '-m', 'acr.training.event',
                "--config", "configs/p2/p2-mixture25.json",
                "--output", str(mixture_run), "--device", "cuda"], check=True)
print((mixture_run / "metrics.json").read_text())


In [ ]:
keys = ["event_accuracy", "rare_macro_event_recall", "macro_quality_event_recall", "best_epoch"]
paths = {
    "natural_0_percent": project / "reports/p2/kaggle-natural/metrics.json",
    "mixture_25_percent": mixture_run / "metrics.json",
    "balanced_50_percent": project / "reports/p2/kaggle-oracle/full/metrics.json",
}
comparison = {"seed": 20261119, "results": {}}
for name, path in paths.items():
    metrics = json.loads(path.read_text())
    comparison["results"][name] = {key: metrics[key] for key in keys}
comparison["note"] = "Single-seed screening; inspect rare precision and E-only successes before judging the tradeoff."
(mixture_run / "sampling_comparison.json").write_text(json.dumps(comparison, indent=2))
print(json.dumps(comparison, indent=2))
print((mixture_run / "per_quality.csv").read_text())


In [ ]:
import shutil
archive = shutil.make_archive("/kaggle/working/p2-mixture25-results", "zip",
                              project, "runs/p2-mixture25-kaggle")
print(archive)
